# Custom CNN: unattended parameter search

Run **after** `colab_smoke_test.ipynb` passed. Use *Run All* and leave the PC awake (a sleeping laptop disconnects the Colab session).

| Stage | What | Budget |
|---|---|---|
| A | one-factor ablations around the baseline, 12 epochs each | `STAGE_A_END_H` |
| B | Optuna study (architecture + optimiser knobs), 10 epochs per trial, bad trials pruned | until `STAGE_B_END_H` |
| C | the best trials trained longer on folds 0 and 1, with test predictions | until `TOTAL_HOURS` |

Everything is exported to Google Drive (`dl_lab2/results/<run>/`) after every run. If the session dies, run the notebook again: finished runs are skipped
and the Optuna study resumes from its backup.

In [1]:
TOTAL_HOURS = 3.5
STAGE_A_END_H = 0.9   # no new ablation starts after this many hours
STAGE_B_END_H = 2.2   # Optuna starts no new trial after this
A_EPOCHS, B_EPOCHS, FINAL_EPOCHS = 12, 10, 30   # ~35 s per epoch measured on the T4 (CPU-bound data loading)
N_TRIALS = 60         # upper bound; the time limit usually stops the study first

REPO_URL = "https://github.com/vumenira/CNN_for_multilabel_classification.git"
BRANCH = "param-tuning"

In [2]:
import os, shutil, subprocess, sys, time, traceback
from pathlib import Path

T0 = time.time()
REPO, ENV_FILE = Path("/content/CNN_for_multilabel_classification"), Path("/content/.env")


def uploaded(name):
    """Where 'Upload to Colab' put a file (the folder is not fixed): shallowest match under /content."""
    hits = [p for p in Path("/content").glob(f"**/{name}") if "drive" not in p.parts]
    return min(hits, key=lambda p: len(p.parts), default=None)


if not ENV_FILE.exists() and uploaded(".env"):
    ENV_FILE = uploaded(".env")
if not REPO.exists():
    archive = uploaded("code.zip")
    if archive:
        shutil.unpack_archive(archive, REPO)
    else:
        url, token = REPO_URL, None
        if ENV_FILE.exists():
            lines = ENV_FILE.read_text().splitlines()
            token = next((l.split("=", 1)[1].strip() for l in lines if l.startswith("GITHUB_TOKEN=")), None)
            if token:
                url = url.replace("https://", f"https://{token}@")
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
        assert r.returncode == 0, "no code.zip uploaded and clone failed: " + (r.stderr.replace(token, "***") if token else r.stderr)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

packages = ["lightning", "timm", "mlflow", "albumentations>=2.0.8", "imagehash",
            "iterative-stratification", "torchmetrics", "optuna", "kaggle", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

from src import colab, results, sweep
from src.data import build_cache

assert colab.mount_drive(), "Drive is not available: stop here, otherwise results are lost with the session"
RESULTS = colab.DRIVE_DIR / "results"
RESULTS.mkdir(exist_ok=True)
colab.load_env(ENV_FILE)
colab.ensure_dataset(colab.DRIVE_DIR)
build_cache("train", 224)
build_cache("test", 224)
print(f"setup took {(time.time() - T0) / 60:.1f} min; results go to {RESULTS}")

Mounted at /content/drive


cache train:   0%|          | 0/12260 [00:00<?, ?it/s]

cache test:   0%|          | 0/18673 [00:00<?, ?it/s]

setup took 3.4 min; results go to /content/drive/MyDrive/dl_lab2/results


## Stage A: one-factor ablations (baseline architecture, 12 epochs)

In [3]:
def ablation(name, **changes):
    return dict(name=f"A-{name}", stage="A", epochs=A_EPOCHS, **changes)


ABLATIONS = [
    ablation("base"),
    ablation("posweight-sqrt", pos_weight="sqrt"),
    ablation("lr-3e-4", lr=3e-4),
    ablation("lr-3e-3", lr=3e-3),
    ablation("wd-1e-2", weight_decay=1e-2),
    ablation("no-batchnorm", use_batchnorm=False),
]

try:
    sweep.run_list(ABLATIONS, RESULTS, deadline=T0 + STAGE_A_END_H * 3600)
except Exception:
    traceback.print_exc()

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


[start] A-base


2026/10/21 12:02:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/21 12:02:13 INFO mlflow.store.db.utils: Updating database tables
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ CNN                        │  2.8 M │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss          │      0 │ train │     0 │
│ 2 │ train_f1  │ MultilabelF1Score          │      0 │ train │     0 │
│ 3 │ val_f1    │ MultilabelF1Score          │      0 │ train │     0 │
│ 4 │ val_ap    │ MultilabelAveragePrecision │      0 │ train │     0 │
└───┴───────────┴────────────────────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.185                                                                     
Modules in train mode: 96                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

MLflow experiment: dl_lab-2; run: 72a4f2bfe834431f9dd2c8b514c30c20
Best checkpoint: /content/CNN_for_multilabel_classification/models/A-no-batchnorm/cnn-epoch=11-val_ap=0.3581.ckpt


/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/matplotlib/scale.py:253: RuntimeWarning: overflow encountered in power
  return np.power(self.base, values)


[done] A-no-batchnorm: val AP 0.3581, F1@0.5 0.2852, 25.1 s/epoch


## Stage B: Optuna (maximises validation macro AP)

In [4]:
study = None
try:
    study = sweep.run_optuna(
        "B", n_trials=N_TRIALS, results_dir=RESULTS,
        overrides=dict(epochs=B_EPOCHS),
        timeout=max(60, T0 + STAGE_B_END_H * 3600 - time.time()),
    )
    print("best trial:", study.best_trial.number, round(study.best_value, 4), study.best_params)
except Exception:
    traceback.print_exc()

[I 2026-10-21 12:33:54,643] A new study created in RDB with name: B
INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automati

[start] B-000


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ CNN                        │  2.8 M │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss          │      0 │ train │     0 │
│ 2 │ train_f1  │ MultilabelF1Score          │      0 │ train │     0 │
│ 3 │ val_f1    │ MultilabelF1Score          │      0 │ train │     0 │
│ 4 │ val_ap    │ MultilabelAveragePrecision │      0 │ train │     0 │
└───┴───────────┴────────────────────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.185                                                                     
Modules in train mode: 96                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

[pruned] B-031: val AP 0.3167, F1@0.5 0.3105, 24.7 s/epoch


[I 2026-10-21 14:12:11,317] Trial 31 pruned. 


best trial: 28 0.7062 {'base_width': 32, 'depth': 'shallow', 'activation': 'relu', 'dropout': 0.19102524128543955, 'lr': 0.00034698579534598526, 'weight_decay': 0.001176187489328776, 'pos_weight': 'sqrt'}


## Stage C: best trials, longer, folds 0 and 1, with test predictions

In [5]:
try:
    if study is None:
        import optuna
        study = optuna.load_study(study_name="B", storage=f"sqlite:///{sweep.tracking.MLRUNS_DIR / 'optuna.db'}")
    finals = sweep.final_configs(study, top_k=2, epochs=FINAL_EPOCHS, folds=(0, 1))
    sweep.run_list(finals, RESULTS, deadline=T0 + TOTAL_HOURS * 3600)
except Exception:
    traceback.print_exc()

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[start] final1-fold0


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ CNN                        │  1.2 M │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss          │      0 │ train │     0 │
│ 2 │ train_f1  │ MultilabelF1Score          │      0 │ train │     0 │
│ 3 │ val_f1    │ MultilabelF1Score          │      0 │ train │     0 │
│ 4 │ val_ap    │ MultilabelAveragePrecision │      0 │ train │     0 │
└───┴───────────┴────────────────────────────┴────────┴───────┴───────┘

Trainable params: 1.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.2 M                                                                                                
Total estimated model params size (MB): 4.911                                                                      
Modules in train mode: 60                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Output()

INFO: Restoring states from the checkpoint path at /content/CNN_for_multilabel_classification/models/final2-fold1/cnn-epoch=28-val_ap=0.7080.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/CNN_for_multilabel_classification/models/final2-fold1/cnn-epoch=28-val_ap=0.7080.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /content/CNN_for_multilabel_classification/models/final2-fold1/cnn-epoch=28-val_ap=0.7080.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at /content/CNN_for_multilabel_classification/models/final2-fold1/cnn-epoch=28-val_ap=0.7080.ckpt
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/matplotlib/scale.py:253: RuntimeWarning: overflow encountered in power
  return np.power(self.base, values)


[done] final2-fold1: val AP 0.7080, F1@0.5 0.5746, 24.5 s/epoch


## Summary

In [6]:
table = results.summarize(RESULTS)
table.to_csv(RESULTS / "summary_all.csv", index=False)
cols = [c for c in ("name", "status", "best_score", "f1_macro_05", "f1_macro_tuned", "sec_per_epoch", "num_parameters",
                    "config.lr", "config.weight_decay", "config.activation", "config.dropout", "config.pos_weight",
                    "config.use_batchnorm", "config.epochs", "config.fold") if c in table]
display(table[cols])
print(f"total time {(time.time() - T0) / 3600:.2f} h; everything is in {RESULTS}")

,name,status,best_score,f1_macro_05,f1_macro_tuned,sec_per_epoch,num_parameters,config.lr,config.weight_decay,config.activation,config.dropout,config.pos_weight,config.use_batchnorm,config.epochs,config.fold
0,final1-fold0,done,0.783574,0.584300,0.734642,24.2,1227685,0.000347,0.001176,relu,0.191025,sqrt,True,30,0
1,final2-fold0,done,0.770498,0.638411,0.736335,24.6,2758517,0.000350,0.001075,relu,0.216038,sqrt,True,30,0
2,final1-fold1,done,0.751200,0.591687,0.729087,25.4,1227685,0.000347,0.001176,relu,0.191025,sqrt,True,30,1
3,final2-fold1,done,0.708003,0.574647,0.686811,24.5,2758517,0.000350,0.001075,relu,0.216038,sqrt,True,30,1
4,B-028,done,0.706212,0.520523,0.634205,24.4,1227685,0.000347,0.001176,relu,0.191025,sqrt,True,10,0
5,B-011,done,0.658097,0.530512,0.636055,24.5,2758517,0.000350,0.001075,relu,0.216038,sqrt,True,10,0
6,B-026,done,0.648681,0.549530,0.634313,24.5,2758517,0.000301,0.000593,relu,0.166096,sqrt,True,10,0
7,B-008,done,0.644433,0.508598,0.652225,24.5,2758517,0.000328,0.000811,relu,0.053946,sqrt,True,10,0
8,B-020,done,0.640234,0.458070,0.593051,24.5,2758517,0.000393,0.007508,relu,0.074339,sqrt,True,10,0
9,B-015,done,0.638869,0.528274,0.625145,24.6,2758517,0.000379,0.003808,relu,0.486548,sqrt,True,10,0


total time 3.85 h; everything is in /content/drive/MyDrive/dl_lab2/results
